In [20]:
##### Cleans FAO nutrient factors data
# decided against using this becuase cotton is not edible, and lots of food crops aren't edible so
# i woudl have to limit it to food, which i don't want ot do

import os
import pandas as pd
from functools import reduce

In [21]:
##### Load data

# Get the current working directory
cd = os.path.dirname(os.getcwd())

# Import data
nutrient_factors = pd.read_excel(f"{cd}/Data/Raw/FAO_production/Nutrient_conversion_table_for_SUA_2024.xlsx", sheet_name='03', skiprows=3)
FAO_codes = pd.read_csv(f"{cd}/Data/Correspondence_tables/FAO_commodity_groups.csv")

# Set save path
save_path = f"{cd}/Data/Clean/Production/FAO_data/FAO_nutrient_factors.csv"

In [22]:
##### Clean nutrient data

# rename columns 
nutrient_factors = nutrient_factors.rename(columns={"Crops and livestock products": "FAO_name"})
nutrient_factors = nutrient_factors.rename(columns={"EDIBLE": "edible_portion"})
nutrient_factors = nutrient_factors.rename(columns={"ENERC ": "kcal_per_100g"})

# convert to kcal_per_tonne
nutrient_factors['kcal_per_edible_tonne'] = nutrient_factors['kcal_per_100g'] * 1e4
nutrient_factors['kcal_per_tonne_production'] = nutrient_factors['kcal_per_edible_tonne'] * nutrient_factors['edible_portion']

# filter columns 
col_to_keep = ['FAO_name', 'kcal_per_tonne_production']
nutrient_factors = nutrient_factors[col_to_keep].copy()

nutrient_factors = nutrient_factors.dropna()

In [23]:
##### Merge with commodity group data 

FAO_codes = FAO_codes.merge(nutrient_factors, on='FAO_name',how='left')

# fill in missing values with group average
FAO_codes["kcal_per_tonne_production"] = (
    FAO_codes["kcal_per_tonne_production"]
    .fillna(
        FAO_codes.groupby("final_group")["kcal_per_tonne_production"]
        .transform("mean")
    )
)